In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 50)

df = pd.read_csv("./Data/Crime_Data.csv")

print(df.shape)

print(df.head(5))

# Check data types and missing values
print(df.info())

# Count missing values in each column
print(df.isna().sum())

In [36]:

# --- DATA CLEANING & PREPARATION ---

# 1. Handling Missing Dates: 
# We drop rows without 'DATE OCC' because our primary analysis is time-based.
df = df[~df["DATE OCC"].isna()]

# 2. Standardizing Categorical Data:
df["VICT SEX"] = df["VICT SEX"].str.upper().str.strip()
df["VICT SEX"] = df["VICT SEX"].replace({"UNKNOWN": np.nan, "X": np.nan})

# 3. Filling Missing Values:
# Instead of dropping rows with missing weapons, we label them 'UNKNOWN' to maintain the integrity of the total crime count. 
df["WEAPON"] = df["WEAPON"].fillna("WEAPON UNKNOWN")

# Final check of the dataset shape after cleaning
print(f"Dataset shape after cleaning: {df.shape}")

# --- FEATURE ENGINEERING ---

# Convert date columns to datetime format
df["DATE RPTD"] = pd.to_datetime(df["DATE RPTD"], errors="coerce")
df["DATE OCC"] = pd.to_datetime(df["DATE OCC"], errors="coerce")

# TIME OCC is in HHMM format, convert to hour
df["TIME OCC"] = pd.to_numeric(df["TIME OCC"], errors="coerce")

df["HOUR OCC"] = (df["TIME OCC"] // 100).astype("Int64")

# Day of week from DATE OCC (0=Monday)
df["DAY OF WEEK"] = df["DATE OCC"].dt.day_name()
df["YEAR"] = df["DATE OCC"].dt.year
df["MONTH"] = df["DATE OCC"].dt.month

Dataset shape after cleaning: (134198, 11)


In [37]:
# 1. Calculate the frequency of each weapon type
weapon_counts = df["WEAPON"].value_counts()

# 2. Select the top 5 most common weapons used in crimes
top_5_weapons = weapon_counts.head(5)

# 6. Display the raw numbers as well
print("Top 5 Weapon Counts:")
print(top_5_weapons)

Top 5 Weapon Counts:
WEAPON
WEAPON UNKNOWN                                    109415
STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)     12600
UNKNOWN WEAPON/OTHER WEAPON                         3915
VERBAL THREAT                                       1702
HAND GUN                                            1486
Name: count, dtype: int64


In [38]:
# --- EXPLORATORY ANALYSIS: CRIME DISTRIBUTION ---

# Create a new column for time of day
def time_of_day(h):
    if pd.isna(h):
        return np.nan
    h = int(h)
    if 5 <= h < 12:
        return "Morning"
    if 12 <= h < 17:
        return "Afternoon"
    if 17 <= h < 21:
        return "Evening"
    return "Night"

df["TIME OF DAY"] = df["HOUR OCC"].apply(time_of_day)

#area of crime grouping
area_counts = (
    df.groupby("AREA NAME")
      .size()
      .sort_values(ascending=False)
      .rename("count")
)

# Crime counts by type
crime_counts = (
    df.groupby("CRIME")
      .size()
      .sort_values(ascending=False)
      .rename("count")
)

print("Area Counts (Top 5):")
print(area_counts.head(5))

print("\nCrime Counts (Top 5):")
print(crime_counts.head(5))

# Time of day grouping
time_counts = (
    df.groupby("TIME OF DAY")
      .size()
      .rename("count")
      .sort_values(ascending=False)
)

print("\nTime of Day Counts:")
print(time_counts)

Area Counts (Top 5):
AREA NAME
Central        10564
Southwest       8645
Pacific         8538
N Hollywood     7704
77th Street     7066
Name: count, dtype: int64

Crime Counts (Top 5):
CRIME
VEHICLE - STOLEN                            21823
BURGLARY FROM VEHICLE                        9737
SHOPLIFTING - PETTY THEFT ($950 & UNDER)     9354
THEFT PLAIN - PETTY ($950 & UNDER)           9246
THEFT OF IDENTITY                            8288
Name: count, dtype: int64

Time of Day Counts:
TIME OF DAY
Afternoon    36545
Night        34915
Evening      31938
Morning      30800
Name: count, dtype: int64


In [39]:
# --- ADVANCED ANALYSIS: STATISTICAL ANOMALIES ---

# 1. Custom NumPy Computation: Z-Score for Area Crime Volume
# Purpose: Identify 'High Density Zones' that are statistically higher than the average crime rate.
area_counts_val = df["AREA NAME"].value_counts()
area_mean = np.mean(area_counts_val) # NumPy mean calculation
area_std = np.std(area_counts_val)   # NumPy standard deviation

# Formula for Z-score: (x - mean) / std_dev
area_z_scores = (area_counts_val - area_mean) / area_std

print("--- Statistical Analysis of Areas ---")
print("Areas with Z-Score > 1.5 (Significant High-Crime Density):")
print(area_z_scores[area_z_scores > 1.5])

# 2. Outlier Analysis: Detecting 'Crime Spikes'
# Purpose: Find specific dates where the number of incidents was an anomaly.
daily_counts = df.groupby("DATE OCC").size()
daily_mean = daily_counts.mean()
daily_std = daily_counts.std()

# We define an outlier as any day 2 standard deviations above the mean crime volume.
outliers = daily_counts[daily_counts > (daily_mean + 2 * daily_std)]

print(f"\n--- Outlier Detection ---")
print(f"Average daily crimes: {daily_mean:.2f} \n")
print(f" outliers detection by days with unusual crime spikes: {len(outliers)}.")
print(outliers.head())

--- Statistical Analysis of Areas ---
Areas with Z-Score > 1.5 (Significant High-Crime Density):
AREA NAME
Central      2.943986
Southwest    1.590362
Pacific      1.514887
Name: count, dtype: float64

--- Outlier Detection ---
Average daily crimes: 97.24 

 outliers detection by days with unusual crime spikes: 103.
DATE OCC
2024-01-01    738
2024-01-02    633
2024-01-03    600
2024-01-04    609
2024-01-05    558
dtype: int64


In [40]:
# --- RELATIONIONAL ANALYSIS: VICTIM DEMOGRAPHICS ---

# Select top 5 crime types for analysis
top_crimes = crime_counts.head(5).index

# Pivot table of crime counts by victim gender
sex_crime = (
    df[df["CRIME"].isin(top_crimes)]
      .pivot_table(
          index="CRIME",
          columns="VICT SEX",
          values="AREA",
          aggfunc="count"
      )
)
print("\n--- Crime Distribution by Victim Gender ---")
print(sex_crime)


--- Crime Distribution by Victim Gender ---
VICT SEX                                       F    H       M
CRIME                                                        
BURGLARY FROM VEHICLE                     4158.0  4.0  5260.0
SHOPLIFTING - PETTY THEFT ($950 & UNDER)   362.0  NaN  2703.0
THEFT OF IDENTITY                         4356.0  1.0  3751.0
THEFT PLAIN - PETTY ($950 & UNDER)        4156.0  2.0  4168.0
VEHICLE - STOLEN                            20.0  NaN    64.0


In [41]:
# --- RELATIONSHIP ANALYSIS: TIME VS CRIME TYPE ---

# Pivot data to see how the top 5 crimes fluctuate throughout the day
top_5_crimes = df["CRIME"].value_counts().head(5).index
relationship_data = (
    df[df["CRIME"].isin(top_5_crimes)]
    .groupby(["TIME OF DAY", "CRIME"])
    .size()
    .unstack()
)
print("\n--- Crime Distribution by Time of Day ---")
print(relationship_data)

# Calculate proportions of crimes by time of day
total_crimes = len(df)
time_counts_array = df["TIME OF DAY"].value_counts().reindex(["Morning","Afternoon","Evening","Night"])
time_props = time_counts_array / total_crimes

print("\n--- Proportions of Crimes by Time of Day ---")
print(time_props)


--- Crime Distribution by Time of Day ---
CRIME        BURGLARY FROM VEHICLE  SHOPLIFTING - PETTY THEFT ($950 & UNDER)  \
TIME OF DAY                                                                    
Afternoon                     1742                                      3688   
Evening                       2873                                      3097   
Morning                       1466                                      1973   
Night                         3656                                       596   

CRIME        THEFT OF IDENTITY  THEFT PLAIN - PETTY ($950 & UNDER)  \
TIME OF DAY                                                          
Afternoon                 3192                                3525   
Evening                   1132                                1930   
Morning                   2737                                2317   
Night                     1227                                1474   

CRIME        VEHICLE - STOLEN  
TIME OF DAY            

In [42]:
# Subgroup comparison: Crime trend by area and time of day

#Identify the top 5 areas for a focused analysis
top_areas = df["AREA NAME"].value_counts().head(5).index

# Group by Area and time of the day to see how crime volume changes across hours for these top areas
time_area_subgroup = (
    df[df["AREA NAME"].isin(top_areas)]
    .groupby(["AREA NAME", "TIME OF DAY"])
    .size()
    .unstack()
)
# Using NumPy to calculate the correlation or trend across time of day for these top areas
hourly_trend = df.groupby("HOUR OCC").size()
mean_crime_rate = np.mean(hourly_trend.values)

print("\n--- Hourly Crime Trend ---")
print(time_area_subgroup)


--- Hourly Crime Trend ---
TIME OF DAY  Afternoon  Evening  Morning  Night
AREA NAME                                      
77th Street       1750     1597     1629   2090
Central           2605     2247     3103   2609
N Hollywood       2276     1845     1624   1959
Pacific           2324     1981     2019   2214
Southwest         2515     2015     1925   2190


In [ ]:
# Visualization to show how crime timing differs by location (Time of Day by Area)
plt.figure(figsize=(10,5))
time_counts.reindex(["Morning","Afternoon","Evening","Night"]).plot(kind="bar", color="orange")
plt.title("Crimes by Time of Day")
plt.xlabel("Time of day")
plt.ylabel("Number of crimes")
plt.tight_layout()
plt.show()

In [44]:
# Visualize the proportions of crimes by time of day
fig, ax = plt.subplots(figsize=(10, 5))

# time_props was calculated in Cell 11
time_props.plot(kind='bar', color=['#4C72B0'], ax=ax)

ax.set_title("Proportion of Crimes by Time of Day", fontsize=14, fontweight='bold')
ax.set_ylabel("Proportion of Total Crimes", fontsize=12)
ax.set_xlabel("Time of Day", fontsize=12)
plt.xticks(rotation=0)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
#bar chart for crime type
plt.figure(figsize=(10,5))
crime_counts.head(5).plot(kind="bar")
plt.title("Top 5 Crime Types")
plt.xlabel("Crime")
plt.ylabel("Number of incidents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
#bar chart for the top 5 areas by crime.
plt.figure(figsize=(10,5))
area_counts.head(5).plot(kind="bar")
plt.title("Top 5 Areas by Number of Crimes")
plt.xlabel("Area name")
plt.ylabel("Number of crimes")
plt.xticks(rotation=0, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization to show how crime timing differs by location (Time of Day by Area)
time_area_subgroup.plot(kind="bar", figsize=(10, 5), edgecolor='black')
plt.title("Crime Distribution: Top 5 Areas by Time of Day", fontsize=14)
plt.ylabel("Number of Incidents")
plt.xlabel("Geographic Area")
plt.xticks(rotation=45)
plt.legend(title="Time of Day")
plt.show()

In [ ]:
#bar chart for top 5 weapons used in crime
plt.figure(figsize=(10, 5))
top_5_weapons.sort_values().plot(kind='barh', color='skyblue', edgecolor='black')

plt.title("Top 5 Weapons Used in Crimes", fontsize=15, pad=20)
plt.xlabel("Number of Incidents", fontsize=12)
plt.ylabel("Weapon Type", fontsize=12)

plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# visualization of relationship analysis between time of day and top crime types
relationship_data.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="viridis")
plt.title("Relationship Between Time of Day and Top Crime Types", fontsize=14)
plt.xlabel("Time of Day")
plt.ylabel("Incident Count")
plt.legend(title="Crime Type", bbox_to_anchor=(1.05, 1))
plt.show()